#### Get the landuse layer data from https://data.gov.sg/datasets?query=masterplan+land+use&resultId=2107

Convert the downloaded geojson file to gpkg for better compatibility with qgis.

```
ogr2ogr -f GPKG -nlt MULTIPOLYGON masterplan2019landuselayer.gpkg MasterPlan2019LandUselayer.geojson
```

Create a new database in sql to work with the gpkg files and add the postgis extensions
```
psql -U <username> -d template1

CREATE DATABASE landuse;

\connect landuse

CREATE EXTENSION postgis;
CREATE EXTENSION postgis_topology;
```

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os
import matplotlib.pyplot as plt

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [5]:
# read the gpkg file
landuse_2019 = Path(BASE_DATASET_PATH / "singapore_data/data_gov/masterplan_2019/masterplan2019landuselayer.gpkg")

landuse_layer_2019 = gpd.read_file(landuse_2019)

In [6]:
landuse_layer_2019.columns

Index(['OBJECTID', 'LU_DESC', 'LU_TEXT', 'GPR', 'WHI_Q_MX', 'GPR_B_MN',
       'INC_CRC', 'FMEL_UPD_D', 'SHAPE.AREA', 'SHAPE.LEN', 'geometry'],
      dtype='object')

#### add the landuse gpkg and subzone boundary gpkg files to sql database

convert the gpkg to EPSG 3414 first
```
ogr2ogr -f GPKG masterplan2019landuselayer_3414.gpkg masterplan2019landuselayer.gpkg \
  -t_srs EPSG:3414
ogr2ogr -f GPKG subzone_boundary_2019_3414.gpkg subzone_boundary_2019.gpkg \
  -t_srs EPSG:3414
```

```
ogr2ogr -f PostgreSQL \
  PG:"dbname=landuse user=yitong" \
  masterplan2019landuselayer_3414.gpkg \
  -nln landuse_2019 \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  G_MP19_LAND_USE_PL

ogr2ogr -f PostgreSQL \
  PG:"dbname=landuse user=yitong" \
  subzone_boundary_2019_3414.gpkg \
  -nln subzone_2019 \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  subzone

UPDATE landuse_2019
SET geom = ST_CollectionExtract(ST_MakeValid(geom), 3)::geometry(MultiPolygon, 3414)
WHERE NOT ST_IsValid(geom);

UPDATE subzone_2019
SET geom = ST_CollectionExtract(ST_MakeValid(geom), 3)::geometry(MultiPolygon, 3414)
WHERE NOT ST_IsValid(geom);
```

```
DROP TABLE IF EXISTS landuse_by_subzone_2019;
CREATE TABLE landuse_by_subzone_2019 AS
SELECT
    lu.LU_DESC,
    
    sz.SUBZONE_N,
    sz.PLN_AREA_N,
    
    ST_Area(
        ST_Intersection(lu.geom, sz.geom)
    ) AS area_m2,
    
    ST_Intersection(lu.geom, sz.geom) AS geometry

FROM landuse_2019 lu 

-- An INNER JOIN ensures that only land use polygons that overlap with a subzone are included.
INNER JOIN subzone_2019 sz
    ON ST_Intersects(lu.geom, sz.geom);
```

Get the output table
```
ogr2ogr -f GPKG landuse_by_subzone_2019.gpkg \
PG:"dbname=landuse user=yitong" landuse_by_subzone_2019
```